In [1]:
import pandas as pd, os, warnings, sys ,shutil, time, ast
from collections import defaultdict
from datetime import datetime

warnings.filterwarnings('ignore')


def parse_dates(date_series,formats=None):
    date_formats = ['%Y-%m-%d', '%Y.%m.%d', '%Y%m%d', '%Y/%m/%d']
    # formats是列表形式，可自定义添加存在的字符串格式
    if formats is None:
        formats = []
    date_formats += formats
    for fmt in date_formats:
        # 尝试解析日期
        parsed_dates = pd.to_datetime(date_series, format=fmt, errors='coerce')
        # 合并成功解析的日期
        date_series = date_series.mask(parsed_dates.notna(), parsed_dates)
    return pd.to_datetime(date_series, errors='coerce')


def print_progress(current, total, bar_length=50,extra_text=''):
    progress = current / total
    block = int(round(bar_length * progress))
    text = f"[{'#' * block + '-' * (bar_length - block)}] {int(progress * 100)}% | ({current}/{total}):{extra_text}"

    # 使用 '\r' 返回行首，'\033[K' 清除行内剩余字符
    sys.stdout.write(f'\r{text}\033[K')
    sys.stdout.flush()


def copy_file(src, dst, tag_=0):       # 从源路径（src)复制一个文件到目标路径(path)，
    tag_update = 0
    if not os.path.exists(src):
        print(f'源文件{src}不存在！无法复制。')
    else:
        if tag_ == 1:  # 当tag_ = 1时，如果文件已存在则进行替换(多为输出的文件)，其中代码和说明文件不进行替换。(除了更新文件)
            if os.path.exists(dst):
                tag_update = 1
                os.remove(dst)
        if not os.path.exists(dst):
            shutil.copyfile(src, dst)
            if tag_ == 1:
                if tag_update == 1:
                    print(f'已更新{src}--->{dst}')
                else:
                    print(f'已复制{src}--->{dst}')
            else:
                print(f'已复制{src}--->{dst}')
        else:
            print(f'{dst}已有文件，不进行复制')


name_err = pd.DataFrame(data=None, columns=['公司名称','表名','产品全称','产品全称_amended'])
err = pd.DataFrame(data=None,columns=['公司名称','产品全称','投资类型','投资策略','投资类型_amended',
                                      '投资策略_amended','投资策略文字描述','处理方式'])

def find_name_err(df):
    new = name_err.copy()
    df_ex = df.drop_duplicates(subset=['产品全称']).reset_index(drop=True).copy()
    df_ex['表名'] = sheet_name
    df_ex['公司名称'] = company_name
    name_df = df_ex['产品全称']
    for index, name in enumerate(name_df):
        if len(name) >= 35 or len(name) <= 10 or any(c in name for c in ['(','（',':','：',')','）','\n']):
            new = pd.concat([new, df_ex[df_ex.index == index][['公司名称','表名','产品全称']]],
                                axis=0, ignore_index=True)
    return new



with open('../参评信息/年份信息.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()
year = lines[0]
yr = int(year)
print()
print('评奖年份:',yr)
print()

with open('../参评信息/采用测试数据.txt','r',encoding='utf-8') as f:
    lines = f.readlines()
    if lines[-1] == '1':
        test = '(测试版)'
    elif lines[-1] == '0':
        test = ''

source_dir = [f'../../原始数据集/{yr}/goldenbull_zqzg/upload_data', f'../../原始数据集/{yr}/goldenbull_zqzg/upload_data_ex']
user_info = pd.read_excel(f'../../原始数据集/{yr}/用户信息表.xlsx',skiprows=[0],dtype={'ID':int})

user_info.info()
if yr == 2024:
    num_dict = {25:'浙商证券', 29:'长江证券', 37:'国泰君安', 46:'广发资管', 47:'太平洋证券', 53:'天风证券', 55:'第一创业',
                58:'万联证券', 64:'东兴证券', 68:'中金公司', 70:'方正证券'}
else:
    num_dict = {} ##待添加

sheetname = ['基础信息', '产品净值', '历任基金经理信息', '规模', '拆分', '分红']

company_dict = {}
error_file = defaultdict(list)

for src in source_dir:
    for subdir, dirs, files in os.walk(src):
        folder_name = os.path.basename(subdir)
        if folder_name in ['1', 'null']:
            continue
        else:
            try:
                num = int(folder_name)
            except:
                continue

            if num not in company_dict.keys():
                company_dict[num] = defaultdict(list)
            if '公司名称' not in company_dict[num].keys():
                company_name_org = str(user_info.loc[user_info['ID'] == num, '公司名称'].values[0])
                # if isinstance(company_name_org, str) and company_name_org != 'nan':
                #     company_name = company_name_org[:4] if '太平洋' not in company_name_org else company_name_org[:5]
                # else:
                #     company_name = num_dict[num]
                if num in num_dict.keys():
                    company_name = num_dict[num]
                else:
                    company_name = company_name_org[:4]
            else:
                company_name = company_dict[num]['公司名称']
            print('\n\n')
            print(f'{num} {company_name}'.center(80, '-'))
            i = 1
            er_file = []
        for file in files:
            # 检查文件名中是否包含'.xls'
            if '.xls' in file.lower():
                file_path = os.path.join(subdir, file)
                xls = pd.ExcelFile(file_path)
                sn = xls.sheet_names
                if len(sn) < 4:
                    if len(sn) == 1:
                        if '净值' in file: sheet_name = '产品净值'
                        elif '经理' in file: sheet_name = '历任基金经理信息'
                        elif '规模' in file: sheet_name = '规模'
                        elif '基础' in file: sheet_name = '基础信息'
                        elif '分红' in file: sheet_name = '分红'
                        elif '拆分' in file: sheet_name = '拆分'
                        df = pd.read_excel(xls,sheet_name='Sheet1')
                        if sheet_name == '基础信息':
                            cols = df.columns.to_list()
                            if '公司名称' not in company_dict[num].keys():
                                # comfirm = input(f'\n输入规定的{num}公司名称,现<<{company_name}>>原<<{company_name_org}>>\n(若已可行，回车即可)：')
                                """confirm = ''
                                ## 注：新一年的评奖时把封印解开！
                                if not comfirm:
                                    pass
                                else:
                                    company_name = comfirm#"""
                                company_dict[num]['公司名称'] = company_name
                            else:
                                company_name = company_dict[num]['公司名称']

                        cols = df.columns.to_list()
                        df['公司名称'] = company_name
                        df = df[['公司名称'] + cols]
                        cols =df.columns.to_list()
                        col = [elm.strip('*').strip() for elm in cols]
                        df.columns = col
                        company_dict[num][sheet_name].append(df)
                    else:
                        error_file[num].append(file)
                        er_file.append(file)
                else:
                    for sheet_name in sn:
                        df = pd.read_excel(xls, sheet_name=sheet_name)

                        # 规范化
                        if '基础' in sheet_name: sheet_name = '基础信息'
                        elif '净值' in sheet_name: sheet_name = '产品净值'
                        elif '经理' in sheet_name: sheet_name = '历任基金经理信息'
                        elif '分红' in sheet_name: sheet_name = '分红'
                        elif '规模' in sheet_name: sheet_name = '规模'
                        elif '拆分' in sheet_name: sheet_name = '拆分'

                        if sheet_name == '基础信息':
                            cols = df.columns.to_list()
                            if '公司名称' not in company_dict[num].keys():
                                # comfirm = input(f'\n输入规定的{num}公司名称,现<<{company_name}>>原<<{company_name_org}>>\n(若已可行，回车即可)：')
                                """confirm = ''
                                ## 注：新一年的评奖时把封印解开！
                                if not comfirm:
                                    pass
                                else:
                                    company_name = comfirm#"""
                                company_dict[num]['公司名称'] = company_name
                            else:
                                company_name = company_dict[num]['公司名称']

                        cols = df.columns.to_list()
                        df['公司名称'] = company_name
                        df = df[['公司名称'] + cols]
                        cols = df.columns.to_list()
                        col = [elm.strip('*').strip() for elm in cols]
                        df.columns = col
                        company_dict[num][sheet_name].append(df)

                    if i + 1 <= len(files):
                        i = i + 1
                    print_progress(i, len(files), 10, f'正在读取:{file}')
                if '分红' not in company_dict[num].keys():
                    print(f'\n{file}少了分红')
                    company_dict[num]['分红'].append(
                        pd.DataFrame(data=None, columns=['公司名称', '产品全称', '分红日期', '每份分红金额(元)']))
                if '拆分' not in company_dict[num].keys():
                    print(f'\n{file}少了拆分')
                    company_dict[num]['拆分'].append(
                        pd.DataFrame(data=None, columns=['公司名称', '产品全称', '拆分日期', '拆分比例']))
            else:
                error_file[num].append(file)
                er_file.append(file)
        if len(er_file) > 0:
            print(f"\n出错文件：{','.join(er_file)}")

for i in company_dict.keys():
    print(company_dict[i]['公司名称'],end=' ')
number = len(company_dict.keys())
print('\n')
print(f'总共有{number}家公司参评'.center(80,'-'))
print()

info_col = ['公司名称','产品全称','成立日期','清盘日期','投资类型','投资策略','投资策略文字描述','是否是被动指数型',
            '是否是QDII','是否分级','是否是海外管理人管理','产品类型','是否大集合','是否公募','预警线','止损线',
            '业绩比较基准','备案号'] \
    if yr == 2024 \
    else ['公司名称','产品全称','成立日期','清盘日期','投资类型','投资策略','投资策略文字描述','是否是被动指数型',
            '是否是QDII','是否分级','是否是海外管理人管理','产品类型','是否大集合','是否公募','预警线','止损线',
            '业绩比较基准','备案号']
nv_col = ['公司名称', '产品全称','日期','单位净值','累计净值','机构占比','券商自持'] if yr == 2024 else \
    ['公司名称','产品全称','日期','单位净值','累计净值','机构占比']
mana_col = ['公司名称','产品全称','开始管理日期','结束管理日期','基金经理姓名']
size_col = ['公司名称','产品全称','统计日期','规模(亿元)','是否为合并数据']
split_col = ['公司名称','产品全称','拆分日期','拆分比例']
bonus_col = ['公司名称','产品全称','分红日期','每份分红金额(元)']

if not os.path.exists(f'../数据源{test}'):
    os.makedirs(f'../数据源{test}')
else:
    shutil.rmtree(f'../数据源{test}')
    os.makedirs(f'../数据源{test}')

if not os.path.exists('../输出文件/其他'):
    os.makedirs('../输出文件/其他')

cate_error = defaultdict(list)

if yr == 2024:
    prod_pairs= [('股票型','股票型'), ('混合型', '积极型混合基金'), ('混合型','偏债型混合基金'), ('混合型','平衡型混合基金'),
                 ('混合型','资产配置基金'), ('债券型', '中长期纯债'), ('债券型', '短期纯债'), ('债券型', '混合债券型（一级）'),
                 ('债券型', '混合债券型（二级）'), ('FOF/MOM型', 'FOF/MOM型')]
elif yr == 2025:
    prod_pairs = [('股票多头型','股票多头型'), ('混合型', '积极型混合基金'), ('混合型','偏债型混合基金'), ('混合型','平衡型混合基金'),
                 ('混合型','资产配置基金'), ('债券型', '中长期纯债'), ('债券型', '短期纯债'), ('债券型', '混合债券型（一级）'),
                 ('债券型', '混合债券型（二级）'), ('FOF型', 'FOF型')]



评奖年份: 2024

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88 entries, 0 to 87
Data columns (total 12 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   ID      88 non-null     int32 
 1   用户名     88 non-null     object
 2   电话      88 non-null     object
 3   邮箱      88 non-null     object
 4   公司代码    88 non-null     object
 5   公司地址    35 non-null     object
 6   公司名称    60 non-null     object
 7   公司备案    13 non-null     object
 8   注册地址    35 non-null     object
 9   办公地址    60 non-null     object
 10  注册时间    60 non-null     object
 11  公司简介    37 non-null     object
dtypes: int32(1), object(11)
memory usage: 8.0+ KB



------------------------------------11 中信期货-------------------------------------
[##########] 100% | (4/4):正在读取:2024证券公司集合资管计划金牛奖机构填报数据表.xlsx-87306454
出错文件：券商集合资产管理人评选指标补充.docx-02426627



------------------------------------13 华宝证券-------------------------------------
[##########] 100% | (3/3):正在读取:2024证券公司集合资管计划金牛奖机构填报数据表-

[##########] 100% | (1/1):正在读取:2024证券公司集合资管计划金牛奖机构填报数据表-华龙证券.xlsx-48942215


------------------------------------66 开源证券-------------------------------------
[##########] 100% | (3/3):正在读取:2024证券公司集合资管计划金牛奖机构填报数据表权益.xlsx-84653041K


------------------------------------67 国海证券-------------------------------------

【产品规模】2024证券公司集合资产管理计划金牛奖机构填报数据表(国海证券).xlsx-77829701少了分红

【产品规模】2024证券公司集合资产管理计划金牛奖机构填报数据表(国海证券).xlsx-77829701少了拆分
[####------] 41% | (5/12):正在读取:【投资策略已更新】【国海证券】2024证券公司集合资产管理计划金牛奖机构填报数据表.xlsx-10946141
出错文件：【产品规模】2024证券公司集合资产管理计划金牛奖机构填报数据表(国海证券).xlsx-77829701,【净值、机构占比等】2024证券公司集合资产管理计划金牛奖机构填报数据表(1).xlsx-57615308,【分红记录】2024证券公司集合资产管理计划金牛奖机构填报数据表(国海证券).xlsx-13701780,【基金经理信息】2024证券公司集合资产管理计划金牛奖机构填报数据表(国海证券).xlsx-17006055,【拆分记录】2024证券公司集合资产管理计划金牛奖机构填报数据表（国海证券）.xlsx-63102060



------------------------------------68 中金公司-------------------------------------
[##########] 100% | (3/3):正在读取:2024证券公司集合资管计划金牛奖机构填报数据表-中金公司.xlsx-65154672


------------------------------------69 江海证券------

[##########] 100% | (1/1):正在读取:2024证券公司集合资管计划金牛奖机构填报数据表-华鑫证券-修正.xlsx-09805245


------------------------------------52 财通证券-------------------------------------

出错文件：固收财通证券资管-2024资管金牛奖填报.pdf-35973712,权益财通证券资管-2024资管金牛奖填报.pdf-61508239,财通证券资管2024资管金牛申报材料.docx-22223143,财通证券资管2024资管金牛申报材料.pdf-10237595



------------------------------------53 天风证券-------------------------------------

出错文件：盖章附件（用印版）.pdf-96087035



------------------------------------57 兴证证券-------------------------------------

出错文件：2024证券公司集合资管计划金牛奖机构填报数据表.pdf-02699646



------------------------------------58 万联证券-------------------------------------

出错文件：万联添利粤升1号.png-82685808,万联聚享添利1号.png-43189603,万联聚享添利3号.png-72090783,万联聚享添利5号.png-53763315,万联证券添利粤信3号.png-34756769,万联证券资产管理（广东）有限公司【万联资管】集合产品WIND同类产品排名2023年12月末截图证明.docx-68411032,万联证券资产管理（广东）有限公司【万联资管】集合产品WIND同类产品排名2023年12月末截图证明——PART1.docx-52022104,万联证券资产管理（广东）有限公司【万联资管】集合产品WIND同类产品排名2023年12月末截图证明——PART2.docx-49746312,万联财星东照月添利1号.png-12624428,产品基本信息——万联资管.pdf-03890073,

In [3]:
i = 1
all_i = len(company_dict.keys())
if not os.path.exists('../输出文件/汇总'):
    os.makedirs('../输出文件/汇总')
with pd.ExcelWriter('../输出文件/汇总/产品汇总.xlsx') as writer_1:
    for num, ins_dict in company_dict.items():
        company_name = company_dict[num]['公司名称']
        print_progress(i, all_i, 50, f'正在处理{company_name}..')
        i += 1
        temp = {}
        # 基础信息
        sheet_name = '基础信息'
        dfs = ins_dict.get(sheet_name)
        info = pd.concat(dfs, axis=0, ignore_index=True)
        if company_name != '中邮证券':
            info = info[info_col]
        else:
            info.rename(columns={'': '产品全称'}, inplace=True)
            # """ 2024年特例
        info = info.drop_duplicates(subset='产品全称', keep='last').reset_index(drop=True).dropna(how='all')
        info['成立日期'] = parse_dates(info['成立日期']).dt.strftime('%Y/%m/%d')
        info['清盘日期'] = parse_dates(info['清盘日期']).dt.strftime('%Y/%m/%d')
        info['清盘日期'] = info['清盘日期'].fillna('2999/12/31')

        info['产品全称'] = info['产品全称'].apply(lambda x: str(x).strip().strip('\n'))

        pairs = pd.Series(zip(info['投资类型'], info['投资策略']))

        for index, pair in enumerate(pairs):
            if pair not in prod_pairs:
                err = pd.concat([err, info[info.index == index][['公司名称', '产品全称', '投资类型',
                                                                 '投资策略', '投资策略文字描述']]],
                                axis=0, ignore_index=True)
        basic = info[['公司名称','产品全称','投资类型','投资策略']].copy()

        name_err = find_name_err(info)

        temp[sheet_name] = info

        # 产品净值
        sheet_name = '产品净值'
        dfs = ins_dict.get(sheet_name)
        nv = pd.concat(dfs, axis=0, ignore_index=True)
        if company_name == '财达证券':
            nv.columns = nv_col
        nv_col_prb = {'产品名称': '产品全称', '累计单位净值': '累计净值', '净值日期': '日期'}
        for name in nv_col_prb.keys():
            if name in nv.columns:
                nv = nv.rename(columns={name: nv_col_prb[name]})
        nv['日期'] = parse_dates(nv['日期'], ['%Y年%m月%d日'])
        nv = nv.dropna(subset=['日期', '产品全称'])
        #nv = nv[nv['日期'].between(f'{yr-5}-1-1',f'{yr-1}-12-31')]
        nv['日期'] = nv['日期'].dt.strftime('%Y/%m/%d')
        nv = nv.drop(nv[nv['产品全称'] == '*产品全称'].index)
        nv['机构占比'] = nv['机构占比'].astype(str).str.replace('-', '').replace('%', '').replace('％', '')
        nv['机构占比'] = pd.to_numeric(nv['机构占比'], errors='coerce')
        nv['机构占比'] = nv['机构占比'].fillna(0)

        if yr == 2024:
            nv['券商自持'] = nv['券商自持'].astype(str).str.replace('-', '').replace('%', '').replace('％', '')
            nv['券商自持'] = pd.to_numeric(nv['券商自持'], errors='coerce')
            nv['券商自持'] = nv['券商自持'].fillna(0)
        nv['产品全称'] = nv['产品全称'].apply(lambda x: str(x).strip())
        nv = nv[nv_col]
        nv = nv.drop_duplicates(subset=['产品全称', '日期'],keep='last').sort_values(
            ['产品全称','日期']).reset_index(drop=True).dropna(how='all')
        layer = nv[['公司名称','产品全称','日期']].copy()
        layer['净值起始日期'] = layer.groupby('产品全称')['日期'].transform('first')
        layer['净值终止日期'] = layer.groupby('产品全称')['日期'].transform('last')
        layer = layer.drop_duplicates(subset=['产品全称']).drop(columns='日期')
        basic = basic.merge(layer, on=['公司名称','产品全称'],how='outer')
        name_err = find_name_err(nv)

        temp[sheet_name] = nv

        # 历任基金经理信息
        sheet_name = '历任基金经理信息'
        dfs = ins_dict.get(sheet_name)
        mag = pd.concat(dfs, axis=0, ignore_index=True)
        mag_col_prb = {'任职起始日': '开始管理日期', '任职终止日': '结束管理日期', '投资主办': '基金经理姓名'}
        for name in mag_col_prb.keys():
            if name in mag.columns:
                mag = mag.rename(columns={name: mag_col_prb[name]})
        mag = mag.dropna(subset=['开始管理日期', '产品全称', '基金经理姓名'])
        mag['产品全称'] = mag['产品全称'].apply(lambda x: str(x).strip())
        mag = mag.drop_duplicates(keep='last').reset_index(drop=True).dropna(how='all')
        mag['开始管理日期'] = parse_dates(mag['开始管理日期']).dt.strftime('%Y/%m/%d')
        mag['结束管理日期'] = parse_dates(mag['结束管理日期']).dt.strftime('%Y/%m/%d')
        mag['结束管理日期'] = mag['结束管理日期'].fillna('2999/12/31')
        mag = mag[mana_col]

        name_err = find_name_err(mag)

        temp[sheet_name] = mag
        # 规模
        sheet_name = '规模'
        dfs = ins_dict.get(sheet_name)
        size = pd.concat(dfs, axis=0, ignore_index=True)
        size = size.drop_duplicates(subset=['产品全称', '统计日期'],keep='last').reset_index(drop=True).dropna(how='all')
        size = size[size_col]
        size = size.dropna(subset=['产品全称', '规模(亿元)', '统计日期'])
        size['产品全称'] = size['产品全称'].apply(lambda x: str(x).strip())
        size['统计日期'] = parse_dates(size['统计日期'])
        layer = size[['公司名称','产品全称', '统计日期', '规模(亿元)']].copy()
        layer['年份'] = layer['统计日期'].dt.year
        size_unique_years = layer.groupby('产品全称')['年份'].unique().reset_index(name='规模覆盖年份')
        size_unique_years['规模覆盖年份'] = size_unique_years['规模覆盖年份'].apply(lambda s: list(s))
        layer = layer.merge(size_unique_years, on='产品全称', how='left')
        layer['平均规模(亿)'] = layer.groupby('产品全称')['规模(亿元)'].transform('mean')
        layer = layer.drop(columns=['年份','规模(亿元)','统计日期']).drop_duplicates(subset='产品全称')
        basic = basic.merge(layer,on=['公司名称','产品全称'],how='outer')

        basic.to_excel(writer_1, sheet_name=f'{num}_{company_name}',index=False)
        size['统计日期'] = size['统计日期'].dt.strftime('%Y/%m/%d')

        name_err = find_name_err(size)

        temp[sheet_name] = size

        # 拆分
        sheet_name = '拆分'
        dfs = ins_dict.get(sheet_name)
        sp = pd.concat(dfs, axis=0, ignore_index=True)
        if not sp.empty:
            sp['产品全称'] = sp['产品全称'].apply(lambda s: str(s).split(':', 1)[1] if ':' in str(s) else s)
            try:
                sp['拆分比例'] = sp['拆分比例'].apply(lambda s: str(s).split('：', 1)[1] if '：' in str(s) else s)
            except:
                pass
            sp = sp.drop_duplicates(keep='last').reset_index(drop=True).dropna(how='all')
            sp = sp.dropna(subset=['拆分日期', '拆分比例', '产品全称'])
            sp = sp.drop(sp[sp['拆分比例'] == '无'].index)
            sp['产品全称'] = sp['产品全称'].apply(lambda x: str(x).strip())
            sp['拆分日期'] = parse_dates(sp['拆分日期']).dt.strftime('%Y/%m/%d')
        sp = sp[split_col]

        name_err = find_name_err(sp)

        temp[sheet_name] = sp

        # 分红
        sheet_name = '分红'
        dfs = ins_dict.get(sheet_name)
        bo = pd.concat(dfs, axis=0, ignore_index=True)
        if not bo.empty:
            bo['产品全称'] = bo['产品全称'].apply(lambda s: str(s).split(':', 1)[1] if ':' in str(s) else s)
            bo = bo.drop_duplicates(keep='last').reset_index(drop=True).dropna(how='all')
            bo = bo.dropna(subset=['产品全称', '分红日期', '每份分红金额(元)'])
            bo['产品全称'] = bo['产品全称'].apply(lambda x: str(x).strip())
            bo['分红日期'] = parse_dates(bo['分红日期']).dt.strftime('%Y/%m/%d')
        bo = bo[bonus_col]

        name_err = find_name_err(bo)

        temp[sheet_name] = bo

        with pd.ExcelWriter(
                f'../数据源(测试版)/{yr}证券公司集合资管计划金牛奖机构填报数据表-{company_name}.xlsx') as writer:
            for sheetn in sheetname:
                for sheet_name, df in temp.items():
                    if sheet_name == sheetn:
                        if sheet_name != '基础信息':
                            df = df.drop(columns='公司名称')
                        df['机构编号'] = num
                        df.to_excel(writer, sheet_name=sheet_name, index=False)

maximum = 0
for lst in error_file.values():
    if maximum < len(lst):
        maximum = len(lst)

for lst in error_file.values():
    for i in range(maximum - len(lst)):
        lst.append('')

error = pd.DataFrame(error_file)

with pd.ExcelWriter('../输出文件/其他/错误.xlsx') as writer:
    error.to_excel(writer, sheet_name='文件读取错误', index=False)
    err.to_excel(writer, sheet_name='策略错误', index=False)
    name_err.to_excel(writer, sheet_name='名称错误', index=False)

print()
copy_file('../资料/代码/2_合并数据表.py', '../step_2 合并数据表/1_合并数据表.py')
copy_file('../资料/说明/数据源说明.docx', '../step_2 合并数据表/0_数据源说明.docx')
print()

print()
print('success')
print()


[##################################################] 100% | (66/66):正在处理财达证券..
../step_2 合并数据表/1_合并数据表.py已有文件，不进行复制
../step_2 合并数据表/0_数据源说明.docx已有文件，不进行复制


success

